In [1]:
# Test with a fake tree
import polars as pl
from igonometree import infer_trees, extract_trees

germline_sequence = "ATGCCGTAATGCCGTAATGCCGTA"*20

all_sequences = [germline_sequence]
for ii in range(200):
    new_sequence = all_sequences[-1]
    all_sequences += [new_sequence[:ii] + 'T' + new_sequence[ii+1:]]
    all_sequences += [new_sequence[:ii] + 'C' + new_sequence[ii+1:]]
    
df = pl.DataFrame(
    {'sequence_alignment': all_sequences,
     'germline_alignment': [germline_sequence[:-5] + 'AAAAA' for a in all_sequences],
     'sequence_id' : [f'Leaf{ii}' for ii, _ in enumerate(all_sequences)],
     'group_id': ['Clonal' for _ in all_sequences]}
)

# test
df = infer_trees(df, n_subsample=50)
tree = extract_trees(df)['Clonal']

100%|███████████████████████████| 1/1 [01:01<00:00, 61.42s/it]


In [2]:
## Test with real sequences
import polars as pl
from igonometree import infer_trees, extract_trees


df = pl.read_csv('example_airr.csv')
df = df.rename({'cdr1fwr3_sequence_alignment': 'sequence_alignment', 
                'cdr1fwr3_germline_alignment': 'germline_alignment', 
                'clonal_family_hilary': 'group_id'})

df = infer_trees(df, n_subsample=50, keep_tmp_files=True)
trees = extract_trees(df)


100%|█████████████████████████| 10/10 [04:52<00:00, 29.23s/it]


In [10]:
# to show the tree, ete4 uses pyQt6 by default, which should be installed (pip install PyQt6)
key = df['group_id'][0]
trees[key].show()